# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/latest/) library.

### Dataset Source
The dataset is defined by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if necessary
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. The `mlcroissant.Dataset` object will help us explore the available data and metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Inspect the metadata object
print('Dataset Title:', dataset.metadata.name)
print('Description:', dataset.metadata.description)
print('Number of available record sets:', len(dataset.metadata.record_sets))

# Optional: Show summary of all top-level metadata
print('\n-- Metadata Summary --')
meta_dict = dataset.metadata.to_json()
for k, v in meta_dict.items():
    if k not in ['record_sets', 'fields', 'columns']:
        print(f"{k}: {v}")

## 2. Data Overview
Explore available record sets, their `@id`, and fields for each. We will use the `@id` fields for all programmatic references as recommended.

In [ ]:
# List available record sets with their @id and name/description
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet: @id={rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id={field.id}, name={field.name}, dataType={getattr(field, 'data_type', None)}")
    print("#"*40)

# Example: list first 2 records for the first RecordSet
if record_sets:
    first_rs_id = record_sets[0].id
    print(f"\nSample records from RecordSet @id={first_rs_id}:\n")
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load records from each record set into a pandas DataFrame for further analysis. All operations use the `@id` fields from the overview above.

In [ ]:
# Prepare DataFrames for each record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = dict()
for rs_id in record_set_ids:
    print(f"Loading records for record set {rs_id}...")
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"- {len(records)} records loaded")

# Preview columns of the first record set
if record_set_ids:
    print("\nColumns in first record set:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    print(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping with respect to the dataset. We'll cast columns to the appropriate types and only use their `@id` in all references.

In [ ]:
# Identify a numeric field from the record set for demonstration.
# Here, we find the first numeric (int/float) column in the first DataFrame.
import numpy as np

example_rs = record_set_ids[0]
df = dataframes[example_rs].copy()

# Try to find a numeric column by dtype or column name
possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or 'age' in col.lower()]
if not possible_numeric_fields:
    # Try to coerce all columns to numeric and use the first that succeeds
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

# Fallback: just pick the first
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    print(f"Using field '{numeric_field_id}' for EDA.")
else:
    print("No obvious numeric field found.")
    numeric_field_id = df.columns[0]

threshold = df[numeric_field_id].quantile(0.5)  # Median as threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouped analysis by a categorical/group field (try 'Sex' or similar field)
group_candidate_fields = [c for c in df.columns if any(x in c.lower() for x in ['sex', 'gender', 'group', 'category', 'anatomical', 'site', 'msi', 'status']) and pd.api.types.is_string_dtype(df[c])]
group_field_id = group_candidate_fields[0] if group_candidate_fields else None
if group_field_id:
    print(f"\nGrouped mean of {numeric_field_id}, by {group_field_id}:")
    print(filtered_df.groupby(group_field_id)[numeric_field_id].mean())
else:
    print("\nNo clear categorical field to group by found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with a group field (if found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If a group field was found, plot group-wise boxplots
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
- We loaded the FAIR² dataset using the Croissant schema and `mlcroissant` library.
- Explored all available record sets, accessed their records and fields using `@id`.
- Applied common filtering, normalization, and grouping strategies for EDA.
- Visualized the key numeric attribute and its distribution; stratified by group where possible.

This notebook provides a reproducible template for analyzing FAIR biomedical datasets described with Croissant schemas &mdash; ensuring clear referencing and reproducibility through semantic `@id` identifiers.